# Deliverable 5 — Progressive Scaling Test 

In [1]:
import os
import sys

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'tests' else os.path.abspath(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from rfbdd import run_pipeline, SELECTORS

print('Available selectors:', list(SELECTORS.keys()))

Available selectors: ['info_gain', 'shapley', 'p_value', 'random_node', 'modal_variable']


## 1. Progressive Scaling on `appendicitis` ($k=3, 5, 7, 10$)

We evaluate progressive scaling across tree prefixes $k \in \{3, 5, 7, 10\}$ and budgets $B \in \{50, 100, 200, 500\}$ for all 5 selectors directly with `run_pipeline`.

In [2]:
# Progressive scaling on appendicitis across trees, budgets, and all 5 selectors
# Printed directly to console with zero file writes (output_dir=None)
configs = [
    (3, [50, 100]),
    (5, [50, 100]),
    (7, [100, 200]),
    (10, [200, 500]),
]

for k, budgets in configs:
    run_pipeline(
        dataset='appendicitis',
        max_trees=k,
        budget=budgets,
        selector='all',
        compute_readouts=False,
        evaluate_robustness=False,
        output_dir=None,
    )

 LAUNCHING GRID PIPELINE: 10 runs
 Datasets (1): ['appendicitis']
 Trees: [3] | Budgets: [50, 100]
 Selectors: ['info_gain', 'shapley', 'p_value', 'random_node', 'modal_variable']
 Output mode: Screen only (no disk persistence)

 GRID PIPELINE EXECUTION SUMMARY
     Dataset  Trees (k)  Budget (B)       Selector  Peak Acc Size  Leaves  Runtime (s)  Completed
appendicitis          3          50      info_gain             23       1       0.0235       True
appendicitis          3          50        shapley             23       1       0.0151       True
appendicitis          3          50        p_value             23       1       0.0151       True
appendicitis          3          50    random_node             23       1       0.0147       True
appendicitis          3          50 modal_variable             23       1       0.0144       True
appendicitis          3         100      info_gain             23       1       0.0162       True
appendicitis          3         100        shapley  

## 2. Multiclass Scaling Validation on `iris`

We validate that `run_pipeline` handles multiclass classification ($C=3$) across budgets and selectors.

In [3]:
# Multiclass validation on iris
run_pipeline(
    dataset='iris',
    max_trees=[3, 5],
    budget=[50, 100],
    selector='all',
    compute_readouts=False,
    evaluate_robustness=False,
    output_dir=None,
)

 LAUNCHING GRID PIPELINE: 20 runs
 Datasets (1): ['iris']
 Trees: [3, 5] | Budgets: [50, 100]
 Selectors: ['info_gain', 'shapley', 'p_value', 'random_node', 'modal_variable']
 Output mode: Screen only (no disk persistence)
[1/20] Running iris (k=3, B=50, selector=info_gain)...
[6/20] Running iris (k=3, B=100, selector=info_gain)...
[11/20] Running iris (k=5, B=50, selector=info_gain)...
[16/20] Running iris (k=5, B=100, selector=info_gain)...

 GRID PIPELINE EXECUTION SUMMARY
Dataset  Trees (k)  Budget (B)       Selector  Peak Acc Size  Leaves  Runtime (s)  Completed
   iris          3          50      info_gain             52       3       0.0509       True
   iris          3          50        shapley             52       3       0.0555       True
   iris          3          50        p_value             52       2       0.0363       True
   iris          3          50    random_node             52       2       0.0456       True
   iris          3          50 modal_variable         

,Dataset,Trees (k),Budget (B),Selector,Peak Acc Size,Leaves,Splits,Runtime (s),Completed
0,iris,3,50,info_gain,52,3,2,0.0509,True
1,iris,3,50,shapley,52,3,2,0.0555,True
2,iris,3,50,p_value,52,2,1,0.0363,True
3,iris,3,50,random_node,52,2,1,0.0456,True
4,iris,3,50,modal_variable,52,2,1,0.0341,True
5,iris,3,100,info_gain,52,1,0,0.0346,True
6,iris,3,100,shapley,52,1,0,0.0351,True
7,iris,3,100,p_value,52,1,0,0.0408,True
8,iris,3,100,random_node,52,1,0,0.0347,True
9,iris,3,100,modal_variable,52,1,0,0.0381,True


## 3. End-to-End Pipeline Showcase (Readouts & MinSwitch Robustness)

Complete pipeline execution on `appendicitis` with model counting, continuous certified volume, feature rankings, and test-set MinSwitch robustness printed directly to screen.

In [4]:
# Showcase single run with readouts and test set robustness
res = run_pipeline(
    dataset='appendicitis',
    max_trees=5,
    budget=100,
    selector='modal_variable',
    compute_readouts=True,
    compute_rankings=True,
    evaluate_robustness=True,
    output_dir=None,
    verbose=True,
)

 PIPELINE EXECUTION: APPENDICITIS
 Trees (D): 5 | Features (F): 7 | Classes (C): 2
 Mode: Budgeted Frontier (B=100, selector=modal_variable)
[*] Discretization complete: 16 binary input bits across 7 features.
[*] Budgeted decomposition finished: 1 leaves in 0.049s (splits: 0, peak size: 82).
[*] Computing certified model counting & continuous volume...
    Class 0: Models =  13312 (20.31%), Volume = 0.2780
    Class 1: Models =  52224 (79.69%), Volume = 0.7220
[*] Computing feature rankings for dominant Class 1...
[*] Evaluating MinSwitch robustness over 22 test samples...
[*] Test Set Robustness Complete in 0.00s:
    Certified Accuracy: 100.00%
    Mean Untargeted Radius: 1.73 (median: 1.00, range: [1, 3])
    Fragile Samples (radius <= 1): 12/22
[*] Total pipeline execution time: 0.82s.
